# Importacion de librerias para NLTK

## Configurar SSL

In [1]:
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

print("✓ Configuración SSL aplicada")


✓ Configuración SSL aplicada


## Importaciones

In [2]:
import sys
import os
import warnings
import nltk
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath("../../../.."))
from src.data.mongo_storage import get_collection, _get_default_collection
from src.data.preprocessor import token_nltk
from src.pos_tagging.nltk_tagger import apply_pos_tagging_nltk

print("✓ Librerías importadas correctamente")

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger', quiet=True)

print("✓ Recursos de NLTK listos")


✓ Librerías importadas correctamente
✓ Recursos de NLTK listos


In [3]:
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
#Si no sirve ejecutar estos comandos
#SOLO PARA VERSION PYTHON 3.13

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kenda\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\kenda\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

## Carga de datos

In [4]:
# Cargar canciones desde MongoDB
import pandas as pd

col = _get_default_collection()
docs = list(col.find({}, {
    "_id": 1, "Song": 1, "Artist": 1, "Genre": 1,
    "Song year": 1, "Lyrics": 1, "Language": 1
}))

df = pd.DataFrame(docs)
print(f"✓ {len(df)} canciones cargadas desde MongoDB")
df.head()


✓ 7935 canciones cargadas desde MongoDB


,_id,Artist,Genre,Language,Lyrics,Song,Song year
0,69c50c3b73a7e31576c65131,buck-65,Hip-Hop,en,Most folks spend their days daydreaming of fin...,craftsmanship,2005
1,69c50c3b73a7e31576c65132,the-elwins,Indie,en,Take your cold hands and put them on my face S...,come-on-out,2012
2,69c50c3b73a7e31576c65133,bullet-for-my-valentine,Metal,en,Are you ready its time for war Well break down...,riot,2013
3,69c50c3b73a7e31576c65134,dream-street,Pop,en,You ask me why I change the color of my hair Y...,that-s-what-girls-do,2007
4,69c50c3b73a7e31576c65135,cassidy,Hip-Hop,en,Do you believe in magic in a young girls heart...,believe-in-a-dollar,2012


## Letras tokenizadas


In [5]:
df= token_nltk(df)
print("Archivo generado correctamente")
df.head()


Archivo generado correctamente


,_id,Artist,Genre,Language,Lyrics,Song,Song year,tokens
0,69c50c3b73a7e31576c65131,buck-65,Hip-Hop,en,Most folks spend their days daydreaming of fin...,craftsmanship,2005,"[Most, folks, spend, their, days, daydreaming,..."
1,69c50c3b73a7e31576c65132,the-elwins,Indie,en,Take your cold hands and put them on my face S...,come-on-out,2012,"[Take, your, cold, hands, and, put, them, on, ..."
2,69c50c3b73a7e31576c65133,bullet-for-my-valentine,Metal,en,Are you ready its time for war Well break down...,riot,2013,"[Are, you, ready, its, time, for, war, Well, b..."
3,69c50c3b73a7e31576c65134,dream-street,Pop,en,You ask me why I change the color of my hair Y...,that-s-what-girls-do,2007,"[You, ask, me, why, I, change, the, color, of,..."
4,69c50c3b73a7e31576c65135,cassidy,Hip-Hop,en,Do you believe in magic in a young girls heart...,believe-in-a-dollar,2012,"[Do, you, believe, in, magic, in, a, young, gi..."


In [6]:
df = apply_pos_tagging_nltk(df)
print("Archivo generado correctamente")
df.head()

pos_tags.nltk guardados en MongoDB: 7935/7935 canciones
Archivo generado correctamente


,_id,Artist,Genre,Language,Lyrics,Song,Song year,tokens,pos_tags
0,69c50c3b73a7e31576c65131,buck-65,Hip-Hop,en,Most folks spend their days daydreaming of fin...,craftsmanship,2005,"[Most, folks, spend, their, days, daydreaming,...","[(Most, JJS), (folks, NNS), (spend, VBP), (the..."
1,69c50c3b73a7e31576c65132,the-elwins,Indie,en,Take your cold hands and put them on my face S...,come-on-out,2012,"[Take, your, cold, hands, and, put, them, on, ...","[(Take, VB), (your, PRP$), (cold, JJ), (hands,..."
2,69c50c3b73a7e31576c65133,bullet-for-my-valentine,Metal,en,Are you ready its time for war Well break down...,riot,2013,"[Are, you, ready, its, time, for, war, Well, b...","[(Are, NNP), (you, PRP), (ready, JJ), (its, PR..."
3,69c50c3b73a7e31576c65134,dream-street,Pop,en,You ask me why I change the color of my hair Y...,that-s-what-girls-do,2007,"[You, ask, me, why, I, change, the, color, of,...","[(You, PRP), (ask, VBP), (me, PRP), (why, WRB)..."
4,69c50c3b73a7e31576c65135,cassidy,Hip-Hop,en,Do you believe in magic in a young girls heart...,believe-in-a-dollar,2012,"[Do, you, believe, in, magic, in, a, young, gi...","[(Do, VBP), (you, PRP), (believe, VB), (in, IN..."


In [7]:
# Verificar que se guardaron correctamente en MongoDB
ejemplo = col.find_one({"pos_tags.nltk": {"$ne": None}})
if ejemplo:
    print(f"Canción: {ejemplo.get('Song')}")
    print(f"POS tags NLTK (primeros 5): {ejemplo['pos_tags']['nltk'][:5]}")
else:
    print("No se encontraron documentos con pos_tags.nltk")


Canción: craftsmanship
POS tags NLTK (primeros 5): [{'token': 'Most', 'tag': 'JJS'}, {'token': 'folks', 'tag': 'NNS'}, {'token': 'spend', 'tag': 'VBP'}, {'token': 'their', 'tag': 'PRP$'}, {'token': 'days', 'tag': 'NNS'}]
